# 📊 Notebook 01 — Análise Exploratória de Dados (EDA)
## Predictfy × Locaweb — FIAP Challenge 2026

**Objetivo:** Explorar os 122.543 incidentes ITSM da Locaweb (jan/2023–dez/2025) e gerar entendimento profundo o suficiente para tomar decisões fundamentadas de feature engineering.

**Contexto de negócio:**
- A Locaweb opera infraestrutura de TI 24×7 com incidentes registrados num sistema ITSM
- Foco nos incidentes com `Entrou para KPI? == SIM` — prioridades P2 (Alta) e P3 (Média)
- **OLA (prazo máximo de resolução):** P2 = 4h (14.400s) · P3 = 12h (43.200s)
- **Meta anual de violações:** P2 = 36–39/ano · P3 = 231–263/ano
- **Resultado 2025:** P2 = 42 violações ❌ (fora da meta) · P3 = 196 ✅ (dentro da meta)

---

---
## 0. Objetivos da EDA

Seguindo a metodologia KDD (Knowledge Discovery in Databases), esta análise exploratória cobre seis objetivos:

1. **Obter e organizar os dados** com rastreabilidade — fonte, período, granularidade e dicionário de dados
2. **Compreender a estrutura do dataset** — dimensões, tipos de variáveis, colunas e seus significados de negócio
3. **Avaliar a qualidade dos dados** — valores ausentes, duplicatas, inconsistências e outliers
4. **Investigar o comportamento univariado** — distribuição de cada variável relevante
5. **Explorar relações multivariadas** — associações entre features e com o target (`KPI Violado?`)
6. **Consolidar achados e decisões** — estratégias de tratamento, feature engineering e modelagem

> **Escopo:** Dataset completo para qualidade + subset KPI (P2+P3, `Entrou para KPI? == SIM`) para análises preditivas.

## 1. Setup e Carregamento dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Estilo dos gráficos
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'figure.dpi': 100,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
})

COLORS = {
    'primary': '#1a73e8',
    'danger': '#d93025',
    'success': '#188038',
    'warning': '#f9ab00',
    'purple': '#7b1fa2',
    'neutral': '#5f6368',
}

PALETTE_KPI = [COLORS['success'], COLORS['danger']]

print('✅ Setup concluído')

In [ ]:
# Carregar dataset
df = pd.read_excel('../data/raw/LW-DATASET.xlsx')
print(f'Shape: {df.shape}')
print(f'Período: {df["Aberto"].min().strftime("%d/%m/%Y")} a {df["Aberto"].max().strftime("%d/%m/%Y")}')
df.head()

In [ ]:
# Visão geral dos tipos e memória
df.info()

In [ ]:
# Estatísticas descritivas das colunas numéricas
df.describe()

---
## 2. Análise de Qualidade dos Dados

Antes de qualquer análise, precisamos entender a integridade dos dados: onde estão os nulos, há duplicatas, os tipos estão corretos?

In [ ]:
# 2.1 — Mapeamento de nulos (absoluto e percentual)
nulos = pd.DataFrame({
    'Nulos': df.isnull().sum(),
    '% Nulo': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('% Nulo', ascending=False)

nulos_filtrado = nulos[nulos['Nulos'] > 0]
print(f'Colunas com nulos: {len(nulos_filtrado)} de {len(df.columns)}')
nulos_filtrado

In [ ]:
# 2.2 — Heatmap visual de nulos
fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(
    df.isnull().T,
    cbar=False,
    cmap=['#e8f0fe', '#d93025'],
    yticklabels=True,
    ax=ax
)
ax.set_title('Mapa de Valores Ausentes por Coluna')
ax.set_xlabel('Incidentes (índice)')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

### 🔍 Investigação: Os nulos de Produto e Categoria são os mesmos incidentes?

Os nulos de Produto (77.935) e Categoria (77.721) são muito próximos. Vamos verificar se são os mesmos registros e como se distribuem por Status e KPI.

In [ ]:
# 2.3 — Cruzamento de nulos: Produto × Categoria × Subcategoria
prod_null = df['Produto'].isnull()
cat_null = df['Categoria'].isnull()
sub_null = df['Subcategoria'].isnull()

print(f'Produto nulo:       {prod_null.sum():>6}')
print(f'Categoria nula:     {cat_null.sum():>6}')
print(f'Subcategoria nula:  {sub_null.sum():>6}')
print(f'Todos 3 nulos:      {(prod_null & cat_null & sub_null).sum():>6}')
print(f'Produto nulo MAS Categoria preenchida: {(prod_null & ~cat_null).sum()}')
print(f'Categoria nula MAS Produto preenchido: {(~prod_null & cat_null).sum()}')

print('\n📋 Distribuição de Status nos registros com Produto nulo:\n')
print(df.loc[prod_null, 'Status'].value_counts())

print('\n📋 Distribuição de "Entrou para KPI?" nos registros com Produto nulo:\n')
print(df.loc[prod_null, 'Entrou para KPI?'].value_counts())

In [ ]:
# 2.4 — Verificar Duração: mínimos e outliers suspeitos
print('=== Estatísticas de Duração (segundos) ===')
print(f'Mínimo:  {df["Duração"].min():>12,}s')
print(f'Máximo:  {df["Duração"].max():>12,}s  ({df["Duração"].max()/3600:,.1f}h)')
print(f'Mediana: {df["Duração"].median():>12,.0f}s')
print(f'Média:   {df["Duração"].mean():>12,.0f}s')

# Quantos incidentes com duração <= 1 segundo?
ultra_rapidos = df[df['Duração'] <= 1]
print(f'\nIncidentes com Duração ≤ 1s: {len(ultra_rapidos)}')
print(f'  Status desses incidentes:')
print(ultra_rapidos['Status'].value_counts().to_string())
print(f'\n  Entrou para KPI?:')
print(ultra_rapidos['Entrou para KPI?'].value_counts().to_string())

In [ ]:
# 2.5 — Verificar duplicatas
dupl = df.duplicated(subset='Número').sum()
print(f'Registros duplicados por "Número": {dupl}')

# Verificar se datas estão corretas (Aberto <= Resolvido <= Encerrado)
mask_resolvido = df['Resolvido'].notna()
inconsist = (df.loc[mask_resolvido, 'Resolvido'] < df.loc[mask_resolvido, 'Aberto']).sum()
print(f'Resolvido antes de Aberto: {inconsist}')

---
## 3. Distribuições e Regras de Negócio

Entender como as variáveis categóricas se distribuem é fundamental para saber quais filtros e agrupamentos fazem sentido.

In [ ]:
# 3.1 — Distribuição de Prioridade
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

prio_counts = df['Prioridade'].value_counts().sort_index()
colors_prio = ['#5f6368', '#d93025', '#f9ab00', '#1a73e8', '#5f6368']

prio_counts.plot(kind='bar', color=colors_prio, edgecolor='white', ax=axes[0])
axes[0].set_title('Volume por Prioridade')
axes[0].set_xlabel('')
axes[0].set_ylabel('Quantidade')
for i, (idx, v) in enumerate(prio_counts.items()):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold', fontsize=10)
axes[0].tick_params(axis='x', rotation=0)

# Tabela percentual
prio_pct = (prio_counts / len(df) * 100).round(2)
prio_df = pd.DataFrame({'Quantidade': prio_counts, '%': prio_pct})
axes[1].axis('off')
table = axes[1].table(
    cellText=prio_df.values,
    colLabels=prio_df.columns,
    rowLabels=prio_df.index,
    loc='center',
    cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.5)
axes[1].set_title('Tabela de Prioridades')

plt.tight_layout()
plt.show()

print('⚠️  P1 tem apenas 1 caso — estatisticamente irrelevante')
print('⚠️  P5 tem 333 casos — nenhum entra para KPI')
print('📌 Foco: P2 (Alta) e P3 (Média) que juntas formam o subset KPI')

In [ ]:
# 3.2 — Distribuição de Status
fig, ax = plt.subplots(figsize=(14, 5))

status_counts = df['Status'].value_counts()
bars = status_counts.plot(kind='barh', color=COLORS['primary'], edgecolor='white', ax=ax)
ax.set_title('Distribuição de Status dos Incidentes')
ax.set_xlabel('Quantidade')

for i, (idx, v) in enumerate(status_counts.items()):
    ax.text(v + 500, i, f'{v:,} ({v/len(df)*100:.1f}%)', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print('📌 "Sem Intervenção" = incidentes resolvidos automaticamente pelo monitoramento')
print('   → Representam alertas que se resolveram sozinhos, não entraram para KPI')

In [ ]:
# 3.3 — Cruzamento: Aberto por × Entrou para KPI?
ct_aberto_kpi = pd.crosstab(
    df['Aberto por'],
    df['Entrou para KPI?'],
    margins=True
)
print('Cruzamento: Aberto por × Entrou para KPI?')
print(ct_aberto_kpi)

print(f'\n📌 Monitoramento abre a maioria dos incidentes')
print(f'   Mas qual % de cada tipo entra para KPI?')

ct_pct = pd.crosstab(df['Aberto por'], df['Entrou para KPI?'], normalize='index') * 100
print(ct_pct.round(1))

In [ ]:
# 3.4 — Distribuição de Entrou para KPI? e KPI Violado?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Entrou para KPI?
kpi_counts = df['Entrou para KPI?'].value_counts()
axes[0].pie(
    kpi_counts, labels=kpi_counts.index, autopct='%1.1f%%',
    colors=[COLORS['primary'], COLORS['neutral']],
    startangle=90, textprops={'fontsize': 12}
)
axes[0].set_title('Entrou para KPI?')

# KPI Violado? (apenas quem entrou)
kpi_sim = df[df['Entrou para KPI?'] == 'SIM']
violado_counts = kpi_sim['KPI Violado?'].value_counts()
axes[1].pie(
    violado_counts, labels=violado_counts.index, autopct='%1.1f%%',
    colors=PALETTE_KPI,
    startangle=90, textprops={'fontsize': 12},
    explode=[0, 0.05]
)
axes[1].set_title('KPI Violado? (dentro do subset KPI)')

plt.tight_layout()
plt.show()

total_kpi = len(kpi_sim)
total_violado = (kpi_sim['KPI Violado?'] == 'SIM').sum()
print(f'Total KPI: {total_kpi:,} | Violações: {total_violado} ({total_violado/total_kpi*100:.2f}%)')

### 3.5 — Análise de Subcategoria (dataset completo)

In [ ]:
# 3.5 — Distribuição de Subcategoria
sub_counts = df['Subcategoria'].value_counts(dropna=False)
print(f'Total de subcategorias únicas (excl. nulos): {df["Subcategoria"].nunique()}')
print(f'Registros com Subcategoria nula: {df["Subcategoria"].isnull().sum()} ({df["Subcategoria"].isnull().mean()*100:.1f}%)')
print()

# Top 20 subcategorias (excluindo nulos)
top_sub = df['Subcategoria'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(14, 6))
top_sub.plot(kind='barh', color=COLORS['purple'], edgecolor='white', ax=ax)
ax.set_title('Top 20 Subcategorias por Volume (dataset completo)')
ax.set_xlabel('Quantidade')
ax.invert_yaxis()
for i, (idx, v) in enumerate(top_sub.items()):
    ax.text(v + 200, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()


---
## 4. Análise do Subset KPI

A partir daqui, focamos nos **25.600 incidentes** que entraram para KPI. Estes são os dados que alimentarão os modelos preditivos.

In [ ]:
# 4.1 — Filtrar subset KPI
kpi = df[df['Entrou para KPI?'] == 'SIM'].copy()
print(f'Shape do subset KPI: {kpi.shape}')
print(f'\nDistribuição por Prioridade:')
print(kpi['Prioridade'].value_counts())

# Separar P2 e P3
p2 = kpi[kpi['Prioridade'] == '2 - Alta']
p3 = kpi[kpi['Prioridade'] == '3 - Média']
print(f'\nP2 (Alta): {len(p2):,} incidentes')
print(f'P3 (Média): {len(p3):,} incidentes')

In [ ]:
# 4.2 — Desbalanceamento: proporção de violações
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (label, subset) in zip(axes, [('Geral KPI', kpi), ('P2 - Alta', p2), ('P3 - Média', p3)]):
    vc = subset['KPI Violado?'].value_counts()
    bars = ax.bar(vc.index, vc.values, color=PALETTE_KPI, edgecolor='white', width=0.5)
    ax.set_title(f'{label}\n({len(subset):,} incidentes)')
    ax.set_ylabel('Quantidade')
    for bar, v in zip(bars, vc.values):
        pct = v / len(subset) * 100
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{v:,}\n({pct:.1f}%)', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Desbalanceamento: KPI Violado?', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

v_p2 = (p2['KPI Violado?'] == 'SIM').sum()
v_p3 = (p3['KPI Violado?'] == 'SIM').sum()
print(f'\n🔴 Implicação para modelagem:')
print(f'   Classe SIM: {v_p2 + v_p3} ({(v_p2+v_p3)/len(kpi)*100:.2f}%)')
print(f'   Classe NAO: {len(kpi) - v_p2 - v_p3} ({(len(kpi)-v_p2-v_p3)/len(kpi)*100:.2f}%)')
print(f'   Razão ~ 1:{(len(kpi)-v_p2-v_p3)//(v_p2+v_p3)}')
print(f'   → Dataset altamente desbalanceado. Acurácia é métrica inútil.')
print(f'   → Necessário SMOTE + métricas: Recall, F1, ROC-AUC')

In [ ]:
# 4.3 — Distribuição de Duração por prioridade
OLA_P2 = 14_400  # 4 horas
OLA_P3 = 43_200  # 12 horas

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (label, subset, ola) in zip(axes, [('P2 - Alta', p2, OLA_P2), ('P3 - Média', p3, OLA_P3)]):
    duracao_h = subset['Duração'] / 3600
    ax.hist(duracao_h, bins=80, color=COLORS['primary'], edgecolor='white', alpha=0.8)
    ax.axvline(ola/3600, color=COLORS['danger'], linewidth=2, linestyle='--', label=f'OLA = {ola/3600:.0f}h')
    ax.set_title(f'Distribuição de Duração — {label}')
    ax.set_xlabel('Duração (horas)')
    ax.set_ylabel('Frequência')
    ax.legend()

plt.tight_layout()
plt.show()

# Percentis
for label, subset, ola in [('P2', p2, OLA_P2), ('P3', p3, OLA_P3)]:
    dur = subset['Duração']
    print(f'\n=== {label} — Percentis de Duração ===')
    for p in [50, 75, 90, 95, 99]:
        val = dur.quantile(p/100)
        print(f'  P{p}: {val:>10,.0f}s  ({val/3600:.1f}h)')
    print(f'  Max: {dur.max():>10,}s  ({dur.max()/3600:,.1f}h)')
    acima = (dur > ola).sum()
    print(f'  Acima do OLA ({ola/3600:.0f}h): {acima} ({acima/len(subset)*100:.2f}%)')

In [ ]:
# 4.4 — Validação: KPI Violado? vs cálculo manual (Duração > OLA)
kpi_check = kpi.copy()
kpi_check['ola_limite'] = kpi_check['Prioridade'].map({
    '2 - Alta': OLA_P2,
    '3 - Média': OLA_P3
})
kpi_check['violacao_manual'] = (kpi_check['Duração'] > kpi_check['ola_limite'])
kpi_check['violacao_dataset'] = (kpi_check['KPI Violado?'] == 'SIM')

# Cruzar
ct = pd.crosstab(
    kpi_check['violacao_dataset'].map({True: 'Dataset=SIM', False: 'Dataset=NAO'}),
    kpi_check['violacao_manual'].map({True: 'Calculado=SIM', False: 'Calculado=NAO'})
)
print('Cruzamento: KPI Violado? (dataset) vs Duração > OLA (calculado)')
print(ct)

n_diverge = ((kpi_check['violacao_dataset'] != kpi_check['violacao_manual']).sum())

print(f'\nDivergências: {n_diverge}')
if n_diverge > 0:
    divergentes = kpi_check[kpi_check['violacao_dataset'] != kpi_check['violacao_manual']]
    print('Exemplos de divergências (Duração > OLA mas KPI = NAO):')
    print(divergentes[['Número', 'Prioridade', 'Duração', 'ola_limite', 'KPI Violado?', 'violacao_manual']].head(10).to_string(index=False))
    print()
    print(f'⚠️  {n_diverge:,} divergências encontradas.')
    print('   → KPI Violado? NÃO é equivalente a (Duração > OLA)')
    print('   → Ver seção 4.4b para investigação detalhada')
    print('   → TARGET CORRETO: usar campo KPI Violado? (ground truth de negócio)')
else:
    print('✅ Sem divergências — KPI Violado? consistente com Duração > OLA')

### 🔴 Investigação das Divergências: Por que `Duração > OLA` ≠ `KPI Violado?`

A validação anterior revelou **3.399 divergências**. O campo `KPI Violado?` é o ground truth de negócio — não é uma simples regra `Duração > OLA`. Precisamos entender o que determina essa diferença.

In [ ]:
# 4.4b — Investigação das 3.399 divergências (Duração > OLA mas KPI = NAO)
divergentes = kpi_check[
    (kpi_check['violacao_dataset'] == False) &
    (kpi_check['violacao_manual'] == True)
].copy()

print(f'Total de divergentes: {len(divergentes)}')
print(f'Duração média dos divergentes: {divergentes["Duração"].mean()/3600:.1f}h')
print(f'Duração mediana dos divergentes: {divergentes["Duração"].median()/3600:.1f}h')
print()

# Distribuição por Status
print('Status dos divergentes:')
print(divergentes['Status'].value_counts().to_string())
print()

# Distribuição por Código de fechamento
print('Código de fechamento dos divergentes:')
print(divergentes['Código de fechamento'].value_counts(dropna=False).head(10).to_string())
print()

# Distribuição por Prioridade
print('Prioridade dos divergentes:')
print(divergentes['Prioridade'].value_counts().to_string())
print()

# Distribuição da duração dos divergentes (em horas)
dur_h = divergentes['Duração'] / 3600
print(f'Duração dos divergentes (horas):')
print(f'  P10: {dur_h.quantile(0.10):.1f}h')
print(f'  P25: {dur_h.quantile(0.25):.1f}h')
print(f'  P50: {dur_h.quantile(0.50):.1f}h')
print(f'  P75: {dur_h.quantile(0.75):.1f}h')
print(f'  P90: {dur_h.quantile(0.90):.1f}h')
print(f'  Max: {dur_h.max():.1f}h')

# Há concentração por grupo/produto?
print()
print('Top 10 grupos com mais divergências:')
print(divergentes['Grupo designado'].value_counts().head(10).to_string())

# Conclusão
print()
print('=' * 60)
print('🔍 CONCLUSÃO:')
print('Os divergentes têm durações MUITO acima do OLA (ver percentis acima).')
print('São incidentes com duração extrema (outliers) que possivelmente:')
print('  1. Estavam em pausa de SLA (ex.: aguardando retorno do cliente)')
print('  2. Foram resolvidos com justificativa/exceção aprovada')
print('  3. Foram encerrados automaticamente sem resolução real')
print()
print('→ O campo KPI Violado? reflete regras de negócio da Locaweb,')
print('  não apenas a comparação Duração > OLA.')
print('→ TARGET CORRETO para o modelo: KPI Violado? (campo original)')
print('  NÃO usar: (Duração > OLA) como target substituto')

---
## 5. Análise Temporal

Entender padrões temporais é essencial para features de sazonalidade e para identificar tendências.

In [ ]:
# 5.1 — Volume mensal por prioridade (subset KPI)
kpi['mes'] = kpi['Aberto'].dt.to_period('M')

vol_mensal = kpi.groupby(['mes', 'Prioridade']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(18, 6))
vol_mensal.plot(kind='bar', stacked=False, color=[COLORS['danger'], COLORS['warning']], 
                edgecolor='white', ax=ax, width=0.7)
ax.set_title('Volume Mensal de Incidentes KPI por Prioridade')
ax.set_xlabel('')
ax.set_ylabel('Quantidade')
ax.legend(title='Prioridade')
ax.tick_params(axis='x', rotation=45)

# Formatar labels do eixo x para mostrar mês/ano
labels = [str(p) for p in vol_mensal.index]
ax.set_xticklabels(labels, rotation=45, ha='right')

plt.tight_layout()
plt.show()

print('📌 P2 só aparece a partir de 2025 no subset KPI — dados limitados para esta prioridade')

In [ ]:
# 5.2 — Heatmap: hora × dia da semana
kpi['hora'] = kpi['Aberto'].dt.hour
kpi['dia_semana'] = kpi['Aberto'].dt.dayofweek  # 0=Seg, 6=Dom
kpi['dia_semana_nome'] = kpi['Aberto'].dt.day_name()

dias_ordem = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dias_pt = ['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta', 'Sábado', 'Domingo']

heat_data = kpi.groupby(['dia_semana', 'hora']).size().unstack(fill_value=0)
heat_data.index = dias_pt

fig, ax = plt.subplots(figsize=(18, 6))
sns.heatmap(heat_data, cmap='YlOrRd', annot=False, fmt='d', linewidths=0.5, ax=ax)
ax.set_title('Heatmap: Volume de Incidentes KPI por Hora × Dia da Semana')
ax.set_xlabel('Hora do dia')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# 5.3 — Volume por dia da semana e por hora do dia
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Por dia da semana
vol_dia = kpi.groupby('dia_semana').size()
vol_dia.index = dias_pt
vol_dia.plot(kind='bar', color=COLORS['primary'], edgecolor='white', ax=axes[0])
axes[0].set_title('Volume por Dia da Semana')
axes[0].set_xlabel('')
axes[0].set_ylabel('Quantidade')
axes[0].tick_params(axis='x', rotation=45)

# Por hora
vol_hora = kpi.groupby('hora').size()
vol_hora.plot(kind='bar', color=COLORS['primary'], edgecolor='white', ax=axes[1])
axes[1].set_title('Volume por Hora do Dia')
axes[1].set_xlabel('Hora')
axes[1].set_ylabel('Quantidade')

plt.tight_layout()
plt.show()

In [ ]:
# 5.4 — Tendência mensal + violações ao longo do tempo
kpi['ano_mes'] = kpi['Aberto'].dt.to_period('M')
kpi['violado'] = (kpi['KPI Violado?'] == 'SIM').astype(int)

mensal = kpi.groupby('ano_mes').agg(
    volume=('Número', 'count'),
    violacoes=('violado', 'sum')
).reset_index()
mensal['ano_mes_dt'] = mensal['ano_mes'].dt.to_timestamp()
mensal['taxa_violacao'] = (mensal['violacoes'] / mensal['volume'] * 100).round(2)

fig, ax1 = plt.subplots(figsize=(18, 6))

ax1.bar(mensal['ano_mes_dt'], mensal['volume'], color=COLORS['primary'], 
        alpha=0.6, label='Volume', width=20)
ax1.set_ylabel('Volume de Incidentes', color=COLORS['primary'])
ax1.set_xlabel('')

ax2 = ax1.twinx()
ax2.plot(mensal['ano_mes_dt'], mensal['violacoes'], color=COLORS['danger'], 
         linewidth=2, marker='o', markersize=5, label='Violações')
ax2.set_ylabel('Violações OLA', color=COLORS['danger'])

ax1.set_title('Volume Mensal e Violações OLA ao Longo do Tempo')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# 5.5 — Violações anuais (comparar com metas)
kpi['ano'] = kpi['Aberto'].dt.year

viol_anual = kpi[kpi['violado'] == 1].groupby(['ano', 'Prioridade']).size().unstack(fill_value=0)
print('Violações anuais por prioridade:')
print(viol_anual)

print(f'\n📊 Metas anuais:')
print(f'   P2: 36-39 violações → 2025: {viol_anual.get("2 - Alta", pd.Series({2025:0})).get(2025, 0)} {"❌" if viol_anual.get("2 - Alta", pd.Series()).get(2025, 0) > 39 else "✅"}')
if '3 - Média' in viol_anual.columns:
    for ano in viol_anual.index:
        v = viol_anual.loc[ano, '3 - Média']
        meta = '✅' if 231 <= v <= 263 else '⚠️' if v < 231 else '❌'
        print(f'   P3 {ano}: {v} violações {meta}')

### 5.6 — Representatividade Histórica: Por que 2023/2024 têm poucas violações?

Os dados de violação de 2023 e 2024 são suspeitos (4 e 6 violações vs 196 em 2025). É fundamental entender se o histórico é confiável para treino.

In [ ]:
# 5.6 — Análise de representatividade histórica por ano
kpi['ano'] = kpi['Aberto'].dt.year

vol_ano = kpi.groupby(['ano', 'Prioridade']).agg(
    volume=('Número', 'count'),
    violacoes=('violado', 'sum')
).reset_index()
vol_ano['taxa_%'] = (vol_ano['violacoes'] / vol_ano['volume'] * 100).round(3)

print('Volume e violações por ano e prioridade:')
print(vol_ano.to_string(index=False))
print()

# Volume total por ano (dataset completo, não só KPI)
vol_total_ano = df.groupby(df['Aberto'].dt.year).size()
print('Volume total de incidentes (dataset completo) por ano:')
print(vol_total_ano.to_string())
print()

# Volume KPI por ano
vol_kpi_ano = kpi.groupby('ano').size()
print('Volume KPI (P2+P3) por ano:')
print(vol_kpi_ano.to_string())
print()

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Volume KPI por ano/prioridade
pivot_vol = vol_ano.pivot(index='ano', columns='Prioridade', values='volume').fillna(0)
pivot_vol.plot(kind='bar', ax=axes[0], color=[COLORS['danger'], COLORS['warning']],
               edgecolor='white')
axes[0].set_title('Volume KPI por Ano e Prioridade')
axes[0].set_xlabel('Ano')
axes[0].set_ylabel('Quantidade')
axes[0].tick_params(axis='x', rotation=0)

# Violações KPI por ano/prioridade
pivot_viol = vol_ano.pivot(index='ano', columns='Prioridade', values='violacoes').fillna(0)
pivot_viol.plot(kind='bar', ax=axes[1], color=[COLORS['danger'], COLORS['warning']],
                edgecolor='white')
axes[1].set_title('Violações OLA por Ano e Prioridade')
axes[1].set_xlabel('Ano')
axes[1].set_ylabel('Violações')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

# Diagnóstico
p3_2023 = vol_ano[(vol_ano['ano'] == 2023) & (vol_ano['Prioridade'] == '3 - Média')]['volume'].values
p3_2024 = vol_ano[(vol_ano['ano'] == 2024) & (vol_ano['Prioridade'] == '3 - Média')]['volume'].values
p3_2025 = vol_ano[(vol_ano['ano'] == 2025) & (vol_ano['Prioridade'] == '3 - Média')]['volume'].values
print()
print('=' * 60)
print('🔍 DIAGNÓSTICO DE REPRESENTATIVIDADE:')
if len(p3_2023) > 0:
    print(f'  P3 2023: {int(p3_2023[0]):,} incidentes — KPI foi monitorado neste ano?')
if len(p3_2024) > 0:
    print(f'  P3 2024: {int(p3_2024[0]):,} incidentes — KPI foi monitorado neste ano?')
if len(p3_2025) > 0:
    print(f'  P3 2025: {int(p3_2025[0]):,} incidentes — base de treino principal')
print()
print('📌 RECOMENDAÇÃO PARA MODELAGEM:')
print('  - O campo KPI Violado? em 2023/2024 tem muito poucas violações vs volume')
print('  - Isso indica que o monitoramento de KPI P3 foi implementado em 2025')
print('  - JANELA DE TREINO RECOMENDADA: 2025 (dados com KPI confiável)')
print('  - 2023/2024 podem ser usados para sazonalidade temporal (Prophet)')
print('  - mas NÃO para classificação de risco OLA (XGBoost)')

---
## 6. Análise de Produtos, Grupos e Categorias

Quais produtos e grupos são mais problemáticos? Onde estão concentradas as violações?

In [ ]:
# 6.1 — Top produtos por volume e taxa de violação (subset KPI)
prod_stats = kpi.groupby('Produto').agg(
    volume=('Número', 'count'),
    violacoes=('violado', 'sum')
).reset_index()
prod_stats['taxa_%'] = (prod_stats['violacoes'] / prod_stats['volume'] * 100).round(2)
prod_stats = prod_stats.sort_values('volume', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Top 15 por volume
top_prod = prod_stats.head(15)
axes[0].barh(top_prod['Produto'], top_prod['volume'], color=COLORS['primary'], edgecolor='white')
axes[0].set_title('Top 15 Produtos por Volume (KPI)')
axes[0].set_xlabel('Quantidade')
axes[0].invert_yaxis()

# Top 15 por taxa de violação (mínimo 10 incidentes)
top_taxa = prod_stats[prod_stats['volume'] >= 10].sort_values('taxa_%', ascending=False).head(15)
colors_taxa = [COLORS['danger'] if t > 5 else COLORS['warning'] if t > 2 else COLORS['success'] for t in top_taxa['taxa_%']]
axes[1].barh(top_taxa['Produto'], top_taxa['taxa_%'], color=colors_taxa, edgecolor='white')
axes[1].set_title('Top 15 Produtos por Taxa de Violação % (min 10 inc.)')
axes[1].set_xlabel('Taxa de Violação (%)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

# Quantos incidentes KPI não têm Produto?
sem_prod = kpi['Produto'].isnull().sum()
print(f'\n⚠️  Incidentes KPI sem Produto: {sem_prod} ({sem_prod/len(kpi)*100:.1f}%)')

In [ ]:
# 6.2 — Top grupos por volume e taxa de violação
grupo_stats = kpi.groupby('Grupo designado').agg(
    volume=('Número', 'count'),
    violacoes=('violado', 'sum')
).reset_index()
grupo_stats['taxa_%'] = (grupo_stats['violacoes'] / grupo_stats['volume'] * 100).round(2)
grupo_stats = grupo_stats.sort_values('volume', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

top_grupo = grupo_stats.head(15)
axes[0].barh(top_grupo['Grupo designado'], top_grupo['volume'], color=COLORS['purple'], edgecolor='white')
axes[0].set_title('Top 15 Grupos por Volume (KPI)')
axes[0].set_xlabel('Quantidade')
axes[0].invert_yaxis()

top_taxa_g = grupo_stats[grupo_stats['volume'] >= 10].sort_values('taxa_%', ascending=False).head(15)
colors_g = [COLORS['danger'] if t > 5 else COLORS['warning'] if t > 2 else COLORS['success'] for t in top_taxa_g['taxa_%']]
axes[1].barh(top_taxa_g['Grupo designado'], top_taxa_g['taxa_%'], color=colors_g, edgecolor='white')
axes[1].set_title('Top 15 Grupos por Taxa de Violação % (min 10 inc.)')
axes[1].set_xlabel('Taxa de Violação (%)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print(grupo_stats.head(10).to_string(index=False))

In [ ]:
# 6.3 — Top categorias por volume e taxa de violação
cat_stats = kpi.dropna(subset=['Categoria']).groupby('Categoria').agg(
    volume=('Número', 'count'),
    violacoes=('violado', 'sum')
).reset_index()
cat_stats['taxa_%'] = (cat_stats['violacoes'] / cat_stats['volume'] * 100).round(2)
cat_stats = cat_stats.sort_values('volume', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

top_cat = cat_stats.head(15)
axes[0].barh(top_cat['Categoria'], top_cat['volume'], color=COLORS['warning'], edgecolor='white')
axes[0].set_title('Top 15 Categorias por Volume (KPI)')
axes[0].set_xlabel('Quantidade')
axes[0].invert_yaxis()

top_taxa_c = cat_stats[cat_stats['volume'] >= 10].sort_values('taxa_%', ascending=False).head(15)
colors_c = [COLORS['danger'] if t > 5 else COLORS['warning'] if t > 2 else COLORS['success'] for t in top_taxa_c['taxa_%']]
axes[1].barh(top_taxa_c['Categoria'], top_taxa_c['taxa_%'], color=colors_c, edgecolor='white')
axes[1].set_title('Top 15 Categorias por Taxa de Violação % (min 10 inc.)')
axes[1].set_xlabel('Taxa de Violação (%)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

sem_cat = kpi['Categoria'].isnull().sum()
print(f'Incidentes KPI sem Categoria: {sem_cat} ({sem_cat/len(kpi)*100:.1f}%)')

---
## 6.4 — Análise de Subcategoria (subset KPI)

Subcategoria foi completamente omitida nas seções anteriores. Aqui verificamos sua distribuição e relação com violações.

In [ ]:
# 6.4 — Subcategoria: volume e taxa de violação no subset KPI
sub_stats = kpi.groupby('Subcategoria').agg(
    volume=('Número', 'count'),
    violacoes=('violado', 'sum')
).reset_index()
sub_stats['taxa_%'] = (sub_stats['violacoes'] / sub_stats['volume'] * 100).round(2)
sub_stats = sub_stats.sort_values('volume', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

top_sub_kpi = sub_stats.head(15)
axes[0].barh(top_sub_kpi['Subcategoria'], top_sub_kpi['volume'],
             color=COLORS['purple'], edgecolor='white')
axes[0].set_title('Top 15 Subcategorias por Volume (KPI)')
axes[0].set_xlabel('Quantidade')
axes[0].invert_yaxis()

top_taxa_sub = sub_stats[sub_stats['volume'] >= 10].sort_values('taxa_%', ascending=False).head(15)
colors_s = [COLORS['danger'] if t > 5 else COLORS['warning'] if t > 2 else COLORS['success']
            for t in top_taxa_sub['taxa_%']]
axes[1].barh(top_taxa_sub['Subcategoria'], top_taxa_sub['taxa_%'],
             color=colors_s, edgecolor='white')
axes[1].set_title('Top 15 Subcategorias por Taxa de Violação % (min 10 inc.)')
axes[1].set_xlabel('Taxa de Violação (%)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

sem_sub = kpi['Subcategoria'].isnull().sum()
print(f'Incidentes KPI sem Subcategoria: {sem_sub} ({sem_sub/len(kpi)*100:.1f}%)')
print()
print(sub_stats.head(10).to_string(index=False))

---
## 6.5 — EDA Bivariada Sistemática: Cramér's V (Associação Categórica × Target)

O **Cramér's V** mede a associação entre duas variáveis categóricas. Valores próximos de 1 indicam forte associação, próximos de 0 indicam ausência de relação. É a métrica correta para avaliar quais features categóricas têm maior poder preditivo em relação ao target binário `KPI Violado?`.

In [ ]:
# 6.5 — Cramér's V para associação entre cada categórica e KPI Violado?
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    """Calcula Cramér's V entre duas séries categóricas."""
    ct = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct, correction=False)[0]
    n = ct.sum().sum()
    r, k = ct.shape
    phi2 = chi2 / n
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)
    denom = min(kcorr - 1, rcorr - 1)
    if denom <= 0:
        return 0.0
    return np.sqrt(phi2corr / denom)

# Features categóricas a avaliar
cat_features = {
    'Prioridade': kpi['Prioridade'],
    'Produto': kpi['Produto'].fillna('DESCONHECIDO'),
    'Categoria': kpi['Categoria'].fillna('DESCONHECIDO'),
    'Subcategoria': kpi['Subcategoria'].fillna('DESCONHECIDO'),
    'Grupo designado': kpi['Grupo designado'],
    'Aberto por': kpi['Aberto por'],
    'Código de fechamento': kpi['Código de fechamento'].fillna('DESCONHECIDO'),
    'hora_cat': kpi['hora'].astype(str) if 'hora' in kpi.columns else kpi['Aberto'].dt.hour.astype(str),
    'dia_semana_cat': kpi['dia_semana'].astype(str) if 'dia_semana' in kpi.columns else kpi['Aberto'].dt.dayofweek.astype(str),
}

target_col = kpi['KPI Violado?']

cv_results = []
for feat_name, feat_series in cat_features.items():
    cv = cramers_v(feat_series, target_col)
    cv_results.append({'Feature': feat_name, "Cramér's V": round(cv, 4)})

cv_df = pd.DataFrame(cv_results).sort_values("Cramér's V", ascending=False)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
colors_cv = [COLORS['danger'] if v > 0.3 else COLORS['warning'] if v > 0.1 else COLORS['neutral']
             for v in cv_df["Cramér's V"]]
bars = ax.barh(cv_df['Feature'], cv_df["Cramér's V"], color=colors_cv, edgecolor='white')
ax.set_title("Cramér's V — Associação de Features Categóricas com KPI Violado?")
ax.set_xlabel("Cramér's V (0=sem associação, 1=associação perfeita)")
ax.invert_yaxis()
ax.axvline(0.3, color=COLORS['danger'], linestyle='--', alpha=0.5, label='Forte (>0.3)')
ax.axvline(0.1, color=COLORS['warning'], linestyle='--', alpha=0.5, label='Moderado (>0.1)')
for bar, v in zip(bars, cv_df["Cramér's V"]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=10)
ax.legend()
plt.tight_layout()
plt.show()

print("\n📊 Ranking de Associação (Cramér's V):")
print(cv_df.to_string(index=False))

---
## 6.6 — Cross-análise: Grupo designado × Produto × KPI Violado?

Identificar combinações de Grupo + Produto com maior taxa de violação.

In [ ]:
# 6.6 — Cross-análise: top combinações Grupo × Produto por taxa de violação
cross = kpi.groupby(['Grupo designado', 'Produto']).agg(
    volume=('Número', 'count'),
    violacoes=('violado', 'sum')
).reset_index()
cross['taxa_%'] = (cross['violacoes'] / cross['volume'] * 100).round(2)

# Top 15 combinações com maior taxa (mínimo 10 incidentes)
top_cross = (cross[cross['volume'] >= 10]
             .sort_values('taxa_%', ascending=False)
             .head(15))

top_cross['label'] = top_cross['Grupo designado'].str[:25] + ' / ' + top_cross['Produto'].str[:20].fillna('N/A')

fig, ax = plt.subplots(figsize=(14, 6))
colors_cx = [COLORS['danger'] if t > 10 else COLORS['warning'] if t > 5 else COLORS['success']
             for t in top_cross['taxa_%']]
ax.barh(top_cross['label'], top_cross['taxa_%'], color=colors_cx, edgecolor='white')
ax.set_title('Top 15 Combinações Grupo × Produto — Taxa de Violação OLA')
ax.set_xlabel('Taxa de Violação (%)')
ax.invert_yaxis()
for i, (_, row) in enumerate(top_cross.iterrows()):
    ax.text(row['taxa_%'] + 0.2, i, f"{row['taxa_%']}% (n={row['volume']})",
            va='center', fontsize=9)
plt.tight_layout()
plt.show()

print("\nTop 15 combinações com maior risco:")
print(top_cross[['Grupo designado', 'Produto', 'volume', 'violacoes', 'taxa_%']].to_string(index=False))

---
## 6.7 — Incidentes Recorrentes: Padrões por `Descrição resumida`

O PDF da Locaweb exige a análise de **incidentes recorrentes**. Aqui usamos a `Descrição resumida` para identificar padrões textuais frequentes e sua relação com violações.

In [ ]:
# 6.7 — Incidentes recorrentes: top descrições e taxa de violação
# Normalizar a descrição (minúsculas, strip)
kpi['desc_norm'] = kpi['Descrição resumida'].str.strip().str.lower()

desc_stats = kpi.groupby('desc_norm').agg(
    volume=('Número', 'count'),
    violacoes=('violado', 'sum')
).reset_index()
desc_stats['taxa_%'] = (desc_stats['violacoes'] / desc_stats['volume'] * 100).round(2)
desc_stats = desc_stats.sort_values('volume', ascending=False)

print(f'Total de descrições únicas: {len(desc_stats):,}')
print(f'Descrições com apenas 1 ocorrência: {(desc_stats["volume"] == 1).sum():,}')
print(f'Top 20 descrições mais recorrentes (com >= 5 ocorrências):')
print()

top_rec = desc_stats[desc_stats['volume'] >= 5].head(20)
print(top_rec[['desc_norm', 'volume', 'violacoes', 'taxa_%']].to_string(index=False))

# Plot: Top 15 mais recorrentes
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

top15 = desc_stats.head(15).copy()
top15['label'] = top15['desc_norm'].str[:60]

axes[0].barh(top15['label'], top15['volume'], color=COLORS['primary'], edgecolor='white')
axes[0].set_title('Top 15 Descrições Mais Recorrentes (subset KPI)')
axes[0].set_xlabel('Quantidade')
axes[0].invert_yaxis()

# Top 15 por taxa de violação (min 5 ocorrências)
top_risk_desc = (desc_stats[desc_stats['volume'] >= 5]
                 .sort_values('taxa_%', ascending=False)
                 .head(15).copy())
top_risk_desc['label'] = top_risk_desc['desc_norm'].str[:60]
colors_rd = [COLORS['danger'] if t > 5 else COLORS['warning'] if t > 2 else COLORS['success']
             for t in top_risk_desc['taxa_%']]
axes[1].barh(top_risk_desc['label'], top_risk_desc['taxa_%'], color=colors_rd, edgecolor='white')
axes[1].set_title('Top 15 Descrições por Taxa de Violação OLA (min 5 ocorrências)')
axes[1].set_xlabel('Taxa de Violação (%)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

# Limpar coluna temporária
kpi.drop(columns=['desc_norm'], inplace=True)

---
## 6.8 — Agrupamento Crítico: Produto × Categoria × Prioridade

O PDF exige explicitamente a análise de **"agrupamentos críticos (produto + categoria + prioridade)"** como eixo analítico. Identificar quais combinações concentram o maior risco operacional.

In [ ]:
# 6.8 — Trivariate: Produto × Categoria × Prioridade × taxa de violação
triv = kpi.dropna(subset=['Produto', 'Categoria']).groupby(
    ['Produto', 'Categoria', 'Prioridade']
).agg(
    volume=('Número', 'count'),
    violacoes=('violado', 'sum')
).reset_index()
triv['taxa_%'] = (triv['violacoes'] / triv['volume'] * 100).round(2)

# Top 20 combinações por taxa de violação (mínimo 10 incidentes)
top_triv = (triv[triv['volume'] >= 10]
            .sort_values('taxa_%', ascending=False)
            .head(20))

print('Top 20 combinações Produto × Categoria × Prioridade por taxa de violação (min 10 inc.):')
print(top_triv[['Produto', 'Categoria', 'Prioridade', 'volume', 'violacoes', 'taxa_%']].to_string(index=False))
print()

# Heatmap: Produto × Categoria (top combinações, somando P2+P3)
hm_data = (kpi.dropna(subset=['Produto', 'Categoria'])
           .groupby(['Produto', 'Categoria'])
           .agg(taxa=('violado', 'mean'))
           .reset_index())

# Pivot: apenas top produtos e categorias (com volume suficiente)
min_vol = kpi.dropna(subset=['Produto', 'Categoria']).groupby(['Produto', 'Categoria']).size()
valid_pairs = min_vol[min_vol >= 10].reset_index()[['Produto', 'Categoria']]
hm_filtered = hm_data.merge(valid_pairs, on=['Produto', 'Categoria'])

if len(hm_filtered) > 0:
    # Selecionar top produtos e categorias por frequência para deixar o heatmap legível
    top_prod_list = (kpi.dropna(subset=['Produto'])['Produto']
                     .value_counts().head(12).index.tolist())
    top_cat_list  = (kpi.dropna(subset=['Categoria'])['Categoria']
                     .value_counts().head(10).index.tolist())
    
    hm_pivot = (hm_filtered[
        hm_filtered['Produto'].isin(top_prod_list) &
        hm_filtered['Categoria'].isin(top_cat_list)
    ].pivot(index='Produto', columns='Categoria', values='taxa') * 100)
    
    if not hm_pivot.empty:
        fig, ax = plt.subplots(figsize=(16, 8))
        sns.heatmap(hm_pivot, cmap='YlOrRd', annot=True, fmt='.1f',
                    linewidths=0.5, ax=ax, cbar_kws={'label': 'Taxa violação (%)'})
        ax.set_title('Heatmap de Taxa de Violação: Produto × Categoria (top 12 × top 10, min 10 inc.)')
        plt.tight_layout()
        plt.show()
    else:
        print('Dados insuficientes para o heatmap com os filtros aplicados.')
else:
    print('Dados insuficientes para o heatmap.')

---
## 6.9 — Interpretação dos Achados Contraintuitivos do Cramér's V

Três resultados do Cramér's V merecem explicação cuidadosa para a apresentação à banca.

In [ ]:
# 6.9 — Explicação dos Cramér's V contraintuitivos

print('=' * 70)
print('1. PRIORIDADE — Cramér\'s V ≈ 0 (resultado contraintuitivo)')
print('=' * 70)

# Mostrar que dentro de cada prioridade a distribuição de violação é similar
for prio, subset, ola in [('P2 Alta', p2, OLA_P2), ('P3 Média', p3, OLA_P3)]:
    viol_pct = subset['violado'].mean() * 100
    print(f'  {prio}: {viol_pct:.2f}% de violações')
print()
print('→ Dentro do subset KPI, P2 e P3 têm taxas de violação parecidas.')
print('  O OLA ESCALA com a prioridade: P2=4h, P3=12h.')
print('  Logo, a prioridade per se não discrimina violação — ela define o threshold.')
print('  Para o modelo XGBoost, prioridade_bin AINDA deve ser feature:')
print('  ela afeta outras interações (ex.: durações absolutas vs relativas).')
print()

print('=' * 70)
print('2. HORA DO DIA — Cramér\'s V = 0 (resultado importante)')
print('=' * 70)
# Taxa de violação por hora
viol_hora = kpi.groupby('hora')['violado'].mean() * 100
print('Taxa de violação por hora (%):\n', viol_hora.round(2).to_string())
print()
print('→ A HORA influencia o VOLUME de incidentes (sazonalidade visível no heatmap).')
print('  Mas NÃO influencia a PROBABILIDADE de violar o OLA uma vez aberto.')
print('  Isso faz sentido: incidentes abertos às 3h têm o mesmo time-to-resolve que às 14h.')
print('  Feature \'hora\' pode ter valor indireto (via capacidade de equipe)')
print('  mas matematicamente não está associada ao target.')
print()

print('=' * 70)
print('3. SUBCATEGORIA — Cramér\'s V = 0.32 (maior, mas 63% nulos)')
print('=' * 70)
# Taxa de violação: DESCONHECIDO vs preenchidos
kpi_temp = kpi.copy()
kpi_temp['sub_flag'] = kpi_temp['Subcategoria'].fillna('DESCONHECIDO')
kpi_temp['sub_flag'] = kpi_temp['sub_flag'].apply(lambda x: 'DESCONHECIDO' if x == 'DESCONHECIDO' else 'Preenchida')

viol_sub_flag = kpi_temp.groupby('sub_flag')['violado'].agg(['count', 'sum', 'mean'])
viol_sub_flag['taxa_%'] = (viol_sub_flag['mean'] * 100).round(3)
print(viol_sub_flag[['count', 'sum', 'taxa_%']])
print()
print('→ O alto Cramér\'s V vem de DUAS fontes:')
print('  a) DESCONHECIDO como categoria em si tem taxa de violação distinta')
print('  b) Entre as subcategorias preenchidas, a variância de taxas é alta')
print('     (algumas subcategorias específicas têm taxas muito altas)')
print()
print('→ Para o modelo: Subcategoria DEVE ser incluída, com nulos = "DESCONHECIDO"')
print('  É a feature com maior poder discriminativo identificado na EDA.')

---
## 7. Investigações Adicionais

Explorações complementares que aprofundam o entendimento dos dados.

In [ ]:
# 7.1 — Duração das violações: quão longe do OLA elas ficam?
violacoes = kpi[kpi['KPI Violado?'] == 'SIM'].copy()
violacoes['ola'] = violacoes['Prioridade'].map({'2 - Alta': OLA_P2, '3 - Média': OLA_P3})
violacoes['excesso_h'] = (violacoes['Duração'] - violacoes['ola']) / 3600

fig, ax = plt.subplots(figsize=(14, 5))
for prio, color in [('3 - Média', COLORS['warning']), ('2 - Alta', COLORS['danger'])]:
    subset = violacoes[violacoes['Prioridade'] == prio]
    if len(subset) > 0:
        ax.hist(subset['excesso_h'], bins=50, alpha=0.7, label=f'{prio} (n={len(subset)})', color=color)

ax.set_title('Distribuição do Excesso de Duração nas Violações (horas acima do OLA)')
ax.set_xlabel('Horas acima do OLA')
ax.set_ylabel('Frequência')
ax.legend()
plt.tight_layout()
plt.show()

print('\nPercentis do excesso (horas acima do OLA):')
for prio in ['2 - Alta', '3 - Média']:
    sub = violacoes[violacoes['Prioridade'] == prio]['excesso_h']
    if len(sub) > 0:
        print(f'  {prio}: mediana={sub.median():.1f}h, P90={sub.quantile(0.9):.1f}h, max={sub.max():.1f}h')

In [ ]:
# 7.2 — Incidentes Pai: qual o efeito no subset KPI?
print('Incidente Pai preenchido no dataset completo:', df['Incidente Pai'].notna().sum())
print('Incidente Pai preenchido no subset KPI:', kpi['Incidente Pai'].notna().sum())

if kpi['Incidente Pai'].notna().sum() > 0:
    com_pai = kpi[kpi['Incidente Pai'].notna()]
    sem_pai = kpi[kpi['Incidente Pai'].isna()]
    
    print(f'\nCom Incidente Pai: {len(com_pai)} | Violações: {com_pai["violado"].sum()} ({com_pai["violado"].mean()*100:.2f}%)')
    print(f'Sem Incidente Pai: {len(sem_pai)} | Violações: {sem_pai["violado"].sum()} ({sem_pai["violado"].mean()*100:.2f}%)')
else:
    print('\n📌 Nenhum incidente KPI tem Incidente Pai preenchido')

In [ ]:
# 7.3 — Código de fechamento: quais tipos têm mais violações?
cod_fech = kpi.dropna(subset=['Código de fechamento']).groupby('Código de fechamento').agg(
    volume=('Número', 'count'),
    violacoes=('violado', 'sum')
).reset_index()
cod_fech['taxa_%'] = (cod_fech['violacoes'] / cod_fech['volume'] * 100).round(2)
cod_fech = cod_fech.sort_values('volume', ascending=False)

fig, ax = plt.subplots(figsize=(14, 5))
x = range(len(cod_fech))
ax.bar(x, cod_fech['volume'], color=COLORS['primary'], alpha=0.7, label='Volume')
ax.set_xticks(x)
ax.set_xticklabels(cod_fech['Código de fechamento'], rotation=45, ha='right')
ax.set_title('Volume por Código de Fechamento (KPI) e Taxa de Violação')
ax.set_ylabel('Volume')

ax2 = ax.twinx()
ax2.plot(x, cod_fech['taxa_%'], color=COLORS['danger'], marker='o', linewidth=2, label='Taxa Violação %')
ax2.set_ylabel('Taxa de Violação (%)', color=COLORS['danger'])

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.show()

print(cod_fech.to_string(index=False))

In [ ]:
# 7.4 — Aberto por: monitoramento vs manual no subset KPI
aberto_stats = kpi.groupby('Aberto por').agg(
    volume=('Número', 'count'),
    violacoes=('violado', 'sum'),
    duracao_mediana=('Duração', 'median')
).reset_index()
aberto_stats['taxa_%'] = (aberto_stats['violacoes'] / aberto_stats['volume'] * 100).round(2)
aberto_stats['duracao_mediana_h'] = (aberto_stats['duracao_mediana'] / 3600).round(2)

print('Aberto por — Estatísticas no subset KPI:')
print(aberto_stats.to_string(index=False))

print('\n📌 Insight: Monitoramento automático detecta mais rápido, mas nem sempre resolve')

In [ ]:
# 7.5 — Outliers extremos de Duração P3
print('=== Investigação de Outliers P3 ===')
p3_sorted = p3.sort_values('Duração', ascending=False)
print(f'Top 10 maiores durações P3:')
print(p3_sorted[['Número', 'Duração', 'Produto', 'Grupo designado', 'KPI Violado?']].head(10).to_string(index=False))

# IQR e threshold
Q1 = p3['Duração'].quantile(0.25)
Q3_val = p3['Duração'].quantile(0.75)
IQR = Q3_val - Q1
upper = Q3_val + 1.5 * IQR
extreme = Q3_val + 3 * IQR

n_outliers = (p3['Duração'] > upper).sum()
n_extremos = (p3['Duração'] > extreme).sum()

print(f'\nIQR: {IQR:,.0f}s | Upper fence (1.5×IQR): {upper:,.0f}s ({upper/3600:.1f}h)')
print(f'Outliers (> 1.5×IQR): {n_outliers}')
print(f'Outliers extremos (> 3×IQR): {n_extremos}')

fig, ax = plt.subplots(figsize=(14, 4))
ax.boxplot(p3['Duração'] / 3600, vert=False, widths=0.5,
           flierprops=dict(marker='.', markerfacecolor=COLORS['danger'], markersize=3))
ax.set_xlabel('Duração (horas)')
ax.set_title('Box Plot — Duração P3 (horas)')
ax.axvline(OLA_P3/3600, color=COLORS['danger'], linestyle='--', label=f'OLA = {OLA_P3/3600:.0f}h')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 7.6 — Correlação entre features numéricas/temporais (subset KPI)
kpi_feat = kpi[['Duração', 'hora', 'dia_semana', 'violado']].copy()
kpi_feat['mes'] = kpi['Aberto'].dt.month

corr = kpi_feat.corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.3f',
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlação entre Features Numéricas (Subset KPI)')
plt.tight_layout()
plt.show()

print('📌 Correlação alta entre Duração e violado é esperada (Duração > OLA → violação)')
print('   → Duração NÃO pode ser feature no modelo (data leakage)')

---
## 8. Conclusões e Decisões para Feature Engineering

### 📌 Features candidatas para o modelo

| Feature | Tipo | Origem | Cramér's V |
|---------|------|--------|------------|
| `Subcategoria` | Categórica | Direto do dataset (nulos → "DESCONHECIDO") | **0.32** ⭐ |
| `Grupo designado` | Categórica | Direto do dataset | 0.11 |
| `Categoria` | Categórica | Direto do dataset (nulos → "DESCONHECIDO") | 0.09 |
| `Código de fechamento` | Categórica | Direto do dataset | 0.08 |
| `Produto` | Categórica | Direto do dataset (nulos → "DESCONHECIDO") | 0.07 |
| `Aberto por` | Categórica | Direto do dataset | 0.04 |
| `Prioridade` | Categórica (binária) | Direto do dataset (P2/P3) | 0.005 ⚠️ |
| `dia_semana` | Numérica/Cíclica | Extraída de `Aberto` | 0.025 |
| `hora` | Numérica/Cíclica | Extraída de `Aberto` | ~0 ⚠️ |
| `mes` | Numérica/Cíclica | Extraída de `Aberto` | — |
| `trimestre` | Numérica | Extraída de `Aberto` | — |
| `lag_1d`, `lag_7d` | Numérica | Volume de incidentes dos últimos 1/7 dias | — |
| `rolling_7d`, `rolling_30d` | Numérica | Média móvel de volume | — |
| `target_ola` | Binária **(TARGET)** | Campo `KPI Violado?` do dataset | — |

> ⚠️ `Prioridade` e `hora` têm Cramér's V ≈ 0, mas são mantidas por valor interativo. Ver seção 6.9.

---

### 🎯 TARGET CORRETO: `KPI Violado?` (campo original do dataset)

A análise da seção 4.4 revelou **3.399 divergências** entre `KPI Violado?` e o cálculo `Duração > OLA`. Isso confirma que o campo `KPI Violado?` reflete **regras de negócio adicionais** da Locaweb (ex.: pausas de SLA, exceções aprovadas) e é o ground truth correto.

> **Decisão de engenharia:** `target_ola = (KPI Violado? == 'SIM').astype(int)` — NÃO usar `(Duração > OLA)` como target.

---

### 📅 JANELA DE TREINAMENTO RECOMENDADA: 2025

A análise histórica (seção 5.6) mostrou que P3 tinha apenas 4 e 6 violações em 2023 e 2024 respectivamente, contra 196 em 2025. Isso indica que o monitoramento de KPI foi implementado sistematicamente apenas em 2025:
- **Para classificação XGBoost:** usar apenas 2025
- **Para previsão de volume (Prophet):** dados de 2023-2025 são válidos para sazonalidade
- **P2:** somente disponível em 2025 no subset KPI

---

### 🚨 Data Leakage — Colunas que NÃO podem ser features

| Coluna | Motivo |
|--------|--------|
| `Duração` | Só conhecida após resolução do incidente |
| `Resolvido` | Só preenchida após resolução |
| `Encerrado` | Só preenchida após encerramento |
| `Código de fechamento` | Só definido ao fechar |
| `Solução` | Só registrada ao resolver |
| `KPI Violado?` | É o próprio target |

---

### 🔧 Estratégia para nulos (MNAR — Missing Not at Random)

Nulos de Produto/Categoria/Subcategoria são concentrados em "Sem Intervenção" (não-KPI). No subset KPI apenas 0.1-0.2% tem nulos. A **ausência de subcategoria é um sinal preditivo** (Cramér's V = 0.32 inclui "DESCONHECIDO" como categoria com comportamento distinto).

**Estratégia:** preencher com `"DESCONHECIDO"` — nunca usar imputação por moda/KNN (violaria a natureza MNAR).

---

### 📐 Outliers de Duração P3

P3 tem 3.460 outliers (> 1.5×IQR) e 3.214 extremos (> 3×IQR), representando ~13% dos incidentes P3. Estes são os mesmos casos que aparecem como divergências na seção 4.4 — incidentes com durações absurdas (milhares de horas) marcados como KPI = NAO. Para o target binário, isso não afeta (o campo `KPI Violado?` é o ground truth). Para features de lag/rolling, aplicar cap no P99.

---

### ⚖️ Desbalanceamento e estratégia

- **248 violações SIM** vs **~25.352 NAO** → razão ~1:102
- Acurácia é inútil como métrica
- **Estratégia:**
  - **SMOTE** apenas no treino (notebook 04)
  - Alternativa: `scale_pos_weight` do XGBoost
  - Métricas: **Recall** (prioridade — capturar violações), **F1-Score**, **ROC-AUC**, **PR-AUC**

---

### 📊 Top 10 Achados para a Banca

1. **Subcategoria é a feature mais preditiva** (Cramér's V = 0.32) — apesar de 63% de nulos no dataset total, a ausência é informativa
2. **Target correto é `KPI Violado?`**, não `Duração > OLA` — 3.399 incidentes excetuados por regras de negócio
3. **P2 violou a meta em 2025** (42 vs 36-39) — o problema é real e urgente
4. **Team07 e Team09** têm as maiores taxas de violação (8.94% e 2.72%) — grupos de risco prioritários
5. **Team03 + lvps** e **Team09 + lssl** têm as maiores taxas combinadas (14.29% e 16.67%)
6. **Manual vs Monitoramento:** incidentes abertos manualmente têm 2.7x mais violações (1.26% vs 0.47%)
7. **Hora do dia não prediz violação** — influencia volume (Prophet), não risco OLA (XGBoost)
8. **Treino recomendado: 2025** — dados de 2023/2024 têm KPI P3 sub-registrado
9. **Incidentes com Duração > OLA mas KPI = NAO:** concentrados em P3 com durações extremas (pausas de SLA)
10. **Sazonalidade temporal** clara no volume: picos em horário comercial, segunda-terça com maior volume

---
*Notebook concluído. Próximo passo: `02_feature_engineering.ipynb` — aplicar todas as decisões acima.*